# 时域反射测量，实际测量与仿真对比

本示例演示了频率域到时域转换的应用，通过比较一个[微带线](#Microstripline)和一个[带有阶跃阻抗部分的微带线](#Stepped-impedance-microstripline)的测量结果和仿真结果来实现。仿真数据由 `skrf` 使用简单的传输线模型生成，该模型用于模拟连接器和每个阻抗部分。为了在测量数据和仿真数据之间实现合理的匹配，需要通过优化来提取介电常数以及连接器的阻抗和延迟。本示例在结尾处给出了[仿真](#Simulation)和[参数优化](#Parameters-optimization)的代码。

## 数据准备

### 设置

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image
from numpy import absolute, log, sum
from scipy.optimize import minimize

import skrf
from skrf.io.general import read_all_networks
from skrf.media import DefinedAEpTandZ0, MLine

skrf.stylely()

### 将数据加载到 skrf 中

测量于 2018 年 2 月 19 日在安立思 MS46524B 20GHz VNA 上进行。设置是 1MHz 到 10GHz 的线性频率扫描，步长为 1MHz，共 10000 个点。如果带宽为 1kHz，输出功率为 0dBm，不进行平滑处理，也不进行平均处理。使用 eCal 校准套件进行完整的双端口校准。关于[时间分辨率](#Time-resolution)和[为了避免混叠响应而设置的范围限制](#Measurement-range-limitation-to-avoid-alias-response)的考虑事项在本文档的末尾。

In [ ]:
#load all measurement and simulation data into a dictionary
meas = read_all_networks('tdr_measurement_vs_simulation/measurement/')
simu = read_all_networks('tdr_measurement_vs_simulation/simulation/')

### 直流点外推

从 1 MHz 到 10 GHz 的测量和模拟数据可用，步长为 1 MHz（谐波扫描），因此我们需要外推直流点。

In [ ]:
names = ['P1-MSL_Stepped_140-P2',
         'P1-MSL_Thru_100-P2']

meas_dc_ext = meas.copy()
simu_dc_ext = simu.copy()
for n in names:
    meas_dc_ext[n] = meas_dc_ext[n].extrapolate_to_dc(kind='linear')
    simu_dc_ext[n] = simu_dc_ext[n].extrapolate_to_dc(kind='linear')


## 微带线

`MSL_Thru_100` 是一种在高度为 $H$ 的衬底上，底部为接地层，长度为 $L1$，宽度为 $W1$，厚度为 $T$ 的铜微带线。| 参数 | 值    || :---      | :---   || $L1$      | 100mm  || $W1$      | 3.0mm  || $T$       | 50µm  || $H$       | 1.5mm || 连接器 | Cinch, 142-0701-851 || 衬底 | FR-4 |

In [ ]:
Image('tdr_measurement_vs_simulation/figures/MSL_100.jpg', width='50%')

### 测量值与仿真结果对比

In [ ]:
plt.figure()
plt.subplot(2,2,1)
plt.title('Time')
meas_dc_ext['P1-MSL_Thru_100-P2'].plot_z_time_step(0, 0, label='meas')
simu_dc_ext['P1-MSL_Thru_100-P2'].plot_z_time_step(0, 0, label='simu')
plt.xlim((-2, 3))

plt.subplot(2,2,2)
plt.title('Frequency')
meas_dc_ext['P1-MSL_Thru_100-P2'].plot_s_db(0, 0, label='meas')
simu_dc_ext['P1-MSL_Thru_100-P2'].plot_s_db(0, 0, label='simu')

plt.subplot(2,2,3)
z0 = 50
t, ymea = meas_dc_ext['P1-MSL_Thru_100-P2'].s11.step_response(window='hamming', pad=0)
ymea[ymea ==  1.] =  1. + 1e-12  # solve numerical singularity
ymea[ymea == -1.] = -1. + 1e-12  # solve numerical singularity
ymea = z0 * (1+ymea) / (1-ymea)
t, ysim = simu_dc_ext['P1-MSL_Thru_100-P2'].s11.step_response(window='hamming', pad=0)
ysim[ysim ==  1.] =  1. + 1e-12  # solve numerical singularity
ysim[ysim == -1.] = -1. + 1e-12  # solve numerical singularity
ysim = z0 * (1+ysim) / (1-ysim)
plt.xlabel('Time (ns)')
plt.ylabel('Relative error (%)')
plt.plot(t*1e9, 100*(ysim-ymea)/ymea)
plt.xlim((-2, 3))

plt.subplot(2,2,4)
delta = simu_dc_ext['P1-MSL_Thru_100-P2'].s_db[:,0,0] - meas_dc_ext['P1-MSL_Thru_100-P2'].s_db[:,0,0]
f = simu_dc_ext['P1-MSL_Thru_100-P2'].f * 1e-9
plt.xlabel('Frequency (GHz)')
plt.ylabel('Delta (dB)')
plt.plot(f, delta)
plt.ylim((-20,20))

plt.tight_layout()
plt.show()

令人惊讶的是，时域结果显示出非常好的吻合度，误差在±2%以内，而频域结果仅在较低频率范围内表现出合理的吻合度。这是因为时域基础形状主要受低频数据的影响。时域数据之间存在一个小的偏移，表明直流点略有不同。由连接器到微带线过渡引起的电感峰值清晰可见，并且其仿真模型与测量结果吻合良好。为了进一步研究，可以测量更大的频率范围，从而提高分辨率，并在更稳定的频率基板上构建被测器件（DUT），但这将增加成本。传输线的横截面也可以提高对实际几何形状的了解，包括制造公差。

## 阶跃阻抗微带线

`MSL_Stepped_100` 是一种阶梯式微带线，由厚度为 $T$ 的铜制成，放置在高度为 $H$ 的基板上，基板下方有接地层。从左到右的四个部分（1、2、3 和 4）的长度为 $Lx$，宽度为 $Wx$，其中 $x$ 是部分的编号。| 参数 | 值    || :--- | :---- || $L1$ | 50mm  || $W1$ | 3.0mm || $L2$ | 20mm  || $W2$ | 8.0mm（电容性）|| $L3$ | 20mm  || $W3$ | 1.0mm（电感性）|| $L4$ | 50mm  || $W4$ | 3.0mm || $T$  | 50umm || $H$  | 1.5mm || 连接器 | Cinch, 142-0701-851 || 基板 | FR-4  |

In [ ]:
Image('tdr_measurement_vs_simulation/figures/MSL_Stepped_140.jpg', width='75%')

### 测量值与仿真值的比较

In [ ]:
plt.figure()
plt.subplot(2,2,1)
plt.title('Time')
meas_dc_ext['P1-MSL_Stepped_140-P2'].plot_z_time_step(0, 0, label='measurement')
simu_dc_ext['P1-MSL_Stepped_140-P2'].plot_z_time_step(0, 0, label='simulation')
plt.xlim((-1, 3))

plt.subplot(2,2,2)
plt.title('Frequency')
meas_dc_ext['P1-MSL_Stepped_140-P2'].plot_s_db(0, 0, label='measurement')
simu_dc_ext['P1-MSL_Stepped_140-P2'].plot_s_db(0, 0, label='simulation')

plt.subplot(2,2,3)
z0 = 50
t, ymea = meas_dc_ext['P1-MSL_Stepped_140-P2'].s11.step_response(window='hamming', pad=0)
ymea[ymea ==  1.] =  1. + 1e-12  # solve numerical singularity
ymea[ymea == -1.] = -1. + 1e-12  # solve numerical singularity
ymea = z0 * (1+ymea) / (1-ymea)
t, ysim = simu_dc_ext['P1-MSL_Stepped_140-P2'].s11.step_response(window='hamming', pad=0)
ysim[ysim ==  1.] =  1. + 1e-12  # solve numerical singularity
ysim[ysim == -1.] = -1. + 1e-12  # solve numerical singularity
ysim = z0 * (1+ysim) / (1-ysim)
plt.xlabel('Time (ns)')
plt.ylabel('Relative error (%)')
plt.plot(t*1e9, 100*(ysim-ymea)/ymea)
plt.xlim((-2, 3))

plt.subplot(2,2,4)
delta = simu_dc_ext['P1-MSL_Stepped_140-P2'].s_db[:,0,0] - meas_dc_ext['P1-MSL_Stepped_140-P2'].s_db[:,0,0]
f = simu_dc_ext['P1-MSL_Stepped_140-P2'].f * 1e-9
plt.xlabel('Frequency (GHz)')
plt.ylabel('Delta (dB)')
plt.plot(f, delta)
plt.ylim((-10,10))

plt.tight_layout()
plt.show()

时域和频域的结果都显示出合理的吻合度，时域结果的误差在 ±5% 以内。频域结果也表现出良好的吻合度，仅在较低频率范围内误差在 ±dB 以内，并且较高频率的吻合度比之前的 [微带线](#Microstripline) 案例中更好。频域结果吻合度更好的一个解释是，阻抗阶跃会产生不连续性，导致更多能量反射回来，从而缩短有效长度并提高精度（增强信号）。阶跃线的电容和电感部分清晰可见，其仿真模型与测量结果吻合良好。第一个连接器的影响被绘图比例掩盖，而第二个连接器的影响则被反射所掩盖。为了进一步改进，可以再次测量更大的频率范围，从而提高分辨率，并在更稳定的频率下构建被测器件，但这会增加成本。传输线的横截面也可以提高对实际几何形状的了解，包括制造公差。最终，可以减小阶跃不连续性，以减小对整体测量的影响（每个不连续性都会影响后续时域信号的形状）。

## 参数优化

### 基于多线法的介电材料有效相对介电常数和损耗角正切的表征

仅需要两条长度不同的线路即可进行介电常数和损耗角的测量。由于我们已经测量了反射信号，因此我们将使用这些信号代替虚拟反射信号。我们也没有开关参数，但由于我们只是提取介电常数和损耗角，而不是进行真正的校准，因此这不会有问题。多线校准算法通过使用多个长度的线路来消除连接器的影响。文档字符串解释说：*在每个频率点，至少应该有一对线路，其相位差不为 0 度或 180 度的倍数，否则校准方程将是奇异的，并且精度会非常差。* 对于所选择的线路组合，这些条件将无法满足，但我们仍然可以获得对介电常数和损耗角的合理估计。

In [ ]:
Image('tdr_measurement_vs_simulation/figures/MSL_100.jpg', width='50%')

In [ ]:
Image('tdr_measurement_vs_simulation/figures/MSL_200.jpg', width='100%')

In [ ]:
Image('tdr_measurement_vs_simulation/figures/MSL_Short_50.jpg', width='25%')

#### 执行 NIST 多线 TRL 算法

忽略“未提供开关项”的警告。

In [ ]:
# Perform NISTMultilineTRL algorithm
line100mm = meas['P1-MSL_Thru_100-P2']
line200mm = meas['P1-MSL_Thru_200-P2']
short50mm = skrf.network.two_port_reflect(meas['P1-MSL_Short_50'], meas['P2-MSL_Short_50'])
measured  = [line100mm, short50mm, line200mm]
Grefls    = [-1]
lengths   = [100e-3, 200e-3] # in meter
offset    = [50e-3] # in meter
cal = skrf.calibration.NISTMultilineTRL(measured, Grefls, lengths, er_est=4.5, refl_offset=offset)

#### 相对介电常数和损耗角正切

`NISTMultilineTRL` 校准可以得到与几何形状相关的、频率相关的有效相对介电常数，它混合了空气和基板的相对介电常数。不幸的是，微带线介质仿真需要给定频率下的基板的单个相对介电常数值，而不是我们从校准中获得的与几何形状相关的、频率相关的有效值。为了克服这个困难，我们将使用一种微带线模型，通过对校准结果进行优化来拟合该模型，使其两个数据集之间的与频率相关的有效相对介电常数的差异最小化。此外，在优化过程中，还会插入一个介电损耗角正切的加权贡献，以最小化测量值与模型值之间的衰减差异。优化结果是 1 GHz 时的介电常数 $\epsilon_r$ 和 $\tan{\delta}$，并将用于介电色散模型。

In [ ]:
# frequency axis
freq  = line100mm.frequency
f     = line100mm.frequency.f
f_ghz = line100mm.frequency.f/1e9

# the physical dimensions of the lines are known by design (neglecting manufacturing tolerances)
W   = 3.00e-3
H   = 1.55e-3
T   = 50e-6
L     = 0.1

# calibration results to compare against
ep_r_mea = cal.er_eff.real
A_mea    = 20/log(10)*cal.gamma.real

# starting values for the optimizer
A     = 0.0
f_A   = 1e9
ep_r0 = 4.5
tanD0 = 0.02
f_epr_tand  = 1e9
x0 = [ep_r0, tanD0]

# function to be minimised
def model(x, freq, ep_r_mea, A_mea, f_ep):
    ep_r, tanD = x[0], x[1]
    m = MLine(frequency=freq, Z0=50, w=W, h=H, t=T,
            ep_r=ep_r, mu_r=1, rho=1.712e-8, tand=tanD, rough=0.15e-6,
             f_low=1e3, f_high=1e12, f_epr_tand=f_ep,
             diel='djordjevicsvensson', disp='kirschningjansen')
    ep_r_mod = m.ep_reff_f.real
    A_mod = m.alpha * 20/log(10)
    return sum((ep_r_mod - ep_r_mea)**2)  + 0.1*sum((A_mod - A_mea)**2)

# run optimizer
res = minimize(model, x0, args=(freq, ep_r_mea, A_mea, f_epr_tand),
               bounds=[(4.0, 5.0), (0.001, 0.1)])

# get the results and print the results
ep_r, tanD = res.x[0], res.x[1]
print(f'epr={ep_r:.3f}, tand={tanD:.4f} at {f_epr_tand * 1e-9:.1f} GHz.')

# build the corresponding media
m = MLine(frequency=freq, Z0=50, w=W, h=H, t=T,
         ep_r=ep_r, mu_r=1, rho=1.712e-8, tand=tanD, rough=0.15e-6,
         f_low=1e3, f_high=1e12, f_epr_tand=f_epr_tand,
         diel='djordjevicsvensson', disp='kirschningjansen')

将基于校准值的测量值与模型值进行对比，以验证数据的合理性。

In [ ]:
plt.figure()
plt.subplot(2,2,1)
plt.xlabel('Frequency [GHz]')
plt.ylabel(r'$\epsilon_{r,eff}$')
plt.plot(f_ghz, ep_r_mea, label='measured')
plt.plot(f_ghz, m.ep_reff_f.real, label='model')
plt.legend()

plt.subplot(2,2,2)
plt.xlabel('Frequency [GHz]')
plt.ylabel('A (dB/m)')
plt.plot(f_ghz, A_mea, label='measured')
A_mod = 20/log(10)*m.alpha
plt.plot(f_ghz, A_mod, label='model')
plt.legend()

plt.subplot(2,2,3)
plt.xlabel('Frequency [GHz]')
plt.ylabel(r'$\epsilon_{r,eff}$ error [%]')
rel_err = 100 * ((ep_r_mea - m.ep_reff_f.real)/ep_r_mea)
plt.plot(f_ghz, rel_err)
plt.ylim((-2,2))

plt.subplot(2,2,4)
plt.xlabel('Frequency [GHz]')
plt.ylabel('$A$ error [%]')
rel_err = 100 * ((A_mea - A_mod)/A_mea)
plt.plot(f_ghz, rel_err)
plt.ylim((-20,10))

plt.tight_layout()
plt.show()

测量结果与模型的拟合度似乎相当合理。在非常低的频率范围之外，$\epsilon_{r,eff}$ 的相对误差保持在 ±1% 以内，并且在大部分频率范围内，$A$ 的相对误差保持在 ±10% 以内。考虑到 $A$ 的形状，使用此模型很难获得更好的结果。

### 连接器效应特性分析

`NISTMultilineTRL` 校准系数包含有关连接器特性的信息，这些信息会在校准过程中进行校正。这些系数可用于拟合基于传输线段的连接器模型。

#### 延迟和衰减

In [ ]:
# extract connector characteristic from port 1 error coefficients
conn = skrf.calibration.error_dict_2_network(cal.coefs, cal.frequency, is_reciprocal=True)[0]

使用线性回归方法，对解调后的相位进行分析，从而估算连接器的延迟。

In [ ]:
# connector delay estimation by linear regression on the unwrapped phase
xlim = 9000 # used to avoid phase jump if any
phi_conn = (np.angle(conn.s[:xlim,1,0]))
z = np.polyfit(f[:xlim], phi_conn, 1)
p = np.poly1d(z)
delay_conn = -z[0]/(2*np.pi)
print(f'Connector delay: {delay_conn * 1e12:.1f} ps')

构建连接器模型，并将其与从校准数据中提取的数据进行比较。

In [ ]:
mc = DefinedAEpTandZ0(m.frequency, ep_r=1, tanD=0.02, Z0=50,
                              f_low=1e3, f_high=1e18, f_ep=f_epr_tand, model='frequencyinvariant')

Z0_conn = 50.0 # the actual connector characteristic impedance will be tuned later
left = mc.line(delay_conn, 's', z0=Z0_conn)
check = mc.thru() ** left ** mc.thru()

plt.figure()
plt.subplot(2,1,1)
conn.plot_s_deg(1, 0, label='measured')
check.plot_s_deg(1, 0, label='model')
plt.ylabel('phase (rad)')
plt.legend()

plt.subplot(2,1,2)
conn.plot_s_db(1, 0, label='Measured')
check.plot_s_db(1, 0, label='Model')
plt.xlabel('Frequency (GHz)')
plt.ylabel('Insertion Loss (dB)')
plt.legend()

plt.tight_layout()
plt.show()

将连接器模型特性与校准结果进行比较，结果显示两者具有合理的吻合度。校准结果显示出一些异常，这些异常与预期的物理行为不符。这是因为校准接近奇异点，原因是直通和线路的相位是 180 度的倍数。可以通过向算法提供更多不同的线路来提高精度，但这些线路目前尚未制造出来。

#### 特性阻抗

现在我们已经估算了连接器的延迟和衰减，那么特性阻抗呢？为了正确地参数化传输线段模型，我们需要这个值。优化算法用于找到使建模结果和测量结果之间的回波损耗差异最小化的特性阻抗。

In [ ]:
s11_ref = conn.s[:,0,0]
x0 = [Z0_conn]

# function to be minimised
def model2(x, mc, delay_conn, s11_ref):
    Z0_mod = x[0]
    conn_mod = mc.line(delay_conn, 's', z0=Z0_mod)
    check = mc.thru(z0 = 50.) ** conn_mod ** mc.thru(z0 = 50.)
    s11_mod = check.s[:,0,0]

    return sum(absolute(s11_ref-s11_mod))

# run optimizer
res = minimize(model2, x0, args=(mc, delay_conn, s11_ref),
               bounds=[(20, 100)])

# get the results and print the results
Z0_conn = res.x[0]
print(f'Z0_conn={Z0_conn:.1f} ohm.')

将建模结果与校准数据进行对比绘制，以验证结果的合理性。

In [ ]:
conn_mod = mc.line(delay_conn, 's', z0=Z0_conn)
check = mc.thru(z0 = 50.) ** conn_mod ** mc.thru(z0 = 50.)

plt.figure()
plt.subplot(2,1,1)
conn.plot_s_db(0,0, label = 'measured')
check.plot_s_db(0,0, label = 'model')


plt.subplot(2,1,2)
plt.plot(check.f*1e-9, (check.s_db[:,0,0]-conn.s_db[:,0,0]))
plt.ylabel('Delta (dB)')
plt.xlabel('Frequency (GHz)')

plt.tight_layout()
plt.show()

在低频和高频范围内，dB 值的差异相当大，但当我们比较测量结果和仿真结果时，会发现时域反射测量结果与该值非常吻合。对于[微带线](#Microstripline)而言，连接器产生的电感峰值得到了很好的体现。

## 仿真

### 频率轴

In [ ]:
freq = skrf.Frequency(1,10e3,10000, 'mhz')

### 具有不同几何形状的介质截面

In [ ]:
# 50 ohm segment
MSL1 = MLine(frequency=freq, z0_port=50, w=W, h=H, t=T,
        ep_r=ep_r, mu_r=1, rho=1.712e-8, tand=tanD, rough=0.15e-6,
        f_low=1e3, f_high=1e12, f_epr_tand=f_epr_tand,
        diel='djordjevicsvensson', disp='kirschningjansen')

# Capacitive segment
MSL2 = MLine(frequency=freq, z0_port=50, w=8.0e-3, h=H, t=T,
        ep_r=ep_r, mu_r=1, rho=1.712e-8, tand=tanD, rough=0.15e-6,
        f_low=1e3, f_high=1e12, f_epr_tand=f_epr_tand,
        diel='djordjevicsvensson', disp='kirschningjansen')

# Inductive segment
MSL3 = MLine(frequency=freq, z0_port=50, w=1.0e-3, h=H, t=T,
        ep_r=ep_r, mu_r=1, rho=1.712e-8, tand=tanD, rough=0.15e-6,
        f_low=1e3, f_high=1e12, f_epr_tand=f_epr_tand,
        diel='djordjevicsvensson', disp='kirschningjansen')

# Connector transmission line media with guessed loss
MCON = DefinedAEpTandZ0(m.frequency, z0_port=50, ep_r=1, tanD=0.025, z0=Z0_conn,
        f_low=1e3, f_high=1e18, f_ep=f_epr_tand, model='frequencyinvariant')

### 模拟的被测设备

In [ ]:
# SMA connector
conn_sma  = MCON.line(delay_conn, 's')


# microstripline
thru_simu = conn_sma ** MSL1.line(100e-3, 'm') ** conn_sma
thru_simu.name = 'P1-MSL_Thru_100-P2'

# stepped impedance microstripline
step_simu = conn_sma \
           ** MSL1.line(50e-3, 'm') \
           ** MSL2.line(20e-3, 'm') \
           ** MSL3.line(20e-3, 'm') \
           ** MSL1.line(50e-3, 'm') \
           ** conn_sma
step_simu.name = 'P1-MSL_Stepped_140-P2'

# write simulated data to .snp files
write_data = False
if write_data:
    step_simu.write_touchstone(dir='tdr_measurement_vs_simulation/simulation/')
    thru_simu.write_touchstone(dir='tdr_measurement_vs_simulation/simulation/')

## 注意事项

### 时间分辨率

在进行直流点外推，并忽略窗口效应后，测量的时域分辨率为：\begin{equation*}Resolution = \frac{1}{f_{span}}\end{equation*}其中，$Resolution$ 是以秒为单位的分辨率，$f_{span}$ 是以赫兹为单位的频率范围。在本例中，$f_{span} = 10 \mskip3mu\mathrm{[GHz]}$，因此：\begin{equation*}Resolution \approx \frac{1}{10^{10}} \approx 100 \quad \mathrm{[ps]}\end{equation*}如果有效相对介电常数约为 3.5，则可以得到以下物理分辨率：\begin{equation*}Resolution_{meter} = \frac{Resolution \dot{} c_0}{\sqrt{\epsilon{}_r}} \approx \frac{10^{-10} \dot{} 3\dot{}10^8}{\sqrt{3.5} } \approx 16 \quad \mathrm{[mm]}\end{equation*}我们将使用反射测量，因此实际范围将除以二，因为信号会来回传播。因此，我们测试设备上的近似距离分辨率将为 8mm。阶梯状阻抗微带线上的不连续性和连接器的影响应该可以清晰地观察到。

### 测量范围限制，以避免出现混叠响应

测量范围应设置为，在不产生重复（混叠）的情况下，能够观察到器件的真实响应。\begin{equation*}Range = \frac{c_0}{\sqrt{\epsilon{}_r} \dot{} \Delta{}_f}\end{equation*}其中，$Range$ 是以米为单位的范围，$c_0$ 是以米/秒为单位的光速，$\Delta{}_f$ 是以赫兹为单位的频率步长，$\epsilon{}_r$ 是被测器件的有效相对介电常数。在给定的测量设置中，并且考虑到 $\Delta{}_f = 1 \mskip3mu\mathrm{[MHz]}$，最坏情况下 $\epsilon{}_r \approx 5.0$，则范围为：\begin{equation*}Range \approx \frac{3\dot{}10^8}{\sqrt{5} \dot{} 10^6} \approx 134 \quad \mathrm{[m]}\end{equation*}FR-4的相对介电常数约为5.0，由于微带线几何结构上方有空气，因此有效的$\epsilon{}_r$会更小，这提供了一个安全裕度，这意味着范围被低估。我们将使用反射测量，因此实际范围除以二，因为信号会来回传播。但是，我们最长的器件长度为200毫米，因此我们可以忽略混叠现象。